In [2]:
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = "../../data/processed/toursense_all_annual_long.csv"
MODELS_DIR = "../../models"
OUTPUT_DIR = "../../data/processed"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# Indicateurs prioritaires du Module 1
INDICATEURS = {
    "wb_arrivals": "Arrivées touristiques (Banque Mondiale)",
    "recettes_devises": "Recettes touristiques (TND)",
    "capacite_lits": "Capacité en lits",
    "taux_occupation": "Taux d'occupation (%)",
    "emplois": "Emplois touristiques directs",
}


In [5]:
# Année de coupure pour la validation temporelle
# (tout ce qui est <= SPLIT_YEAR = train, > SPLIT_YEAR = test)
SPLIT_YEAR = 2019

In [6]:
# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================
def prepare_series(df: pd.DataFrame, key: str) -> pd.DataFrame:
    """Prépare une série au format Prophet (ds, y)."""
    data = df[df["Dataset"] == key].copy()
    if data.empty:
        return pd.DataFrame()

    data["ds"] = pd.to_datetime(data["Annee"].astype(str) + "-12-31")
    data = data.rename(columns={"Valeur": "y"})
    data = data[["ds", "y"]].dropna().sort_values("ds").reset_index(drop=True)
    return data
    
def compute_metrics(y_true, y_pred) -> dict:
    """Calcule MAE, RMSE et MAPE."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # Évite division par zéro
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}
    

def train_prophet(train_df: pd.DataFrame, interval_width: float = 0.90) -> Prophet:
    """Entraîne un modèle Prophet standard."""
    model = Prophet(
        yearly_seasonality=True,
        interval_width=interval_width,
        seasonality_mode="multiplicative",  # adapté aux séries économiques
        changepoint_prior_scale=0.05,       # un peu plus conservateur
    )
    model.fit(train_df)
    return model


In [7]:

# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================
print("=" * 70)
print("TouriSense.AI – Module 1 : Forecasting avec Prophet (version corrigée)")
print("=" * 70)

df = pd.read_csv(DATA_PATH)
print(f"\nDonnées chargées : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print("Indicateurs disponibles :", df["Dataset"].unique().tolist())

TouriSense.AI – Module 1 : Forecasting avec Prophet (version corrigée)

Données chargées : 597 lignes × 4 colonnes
Indicateurs disponibles : ['capacite_region', 'capacite_lits', 'duree_sejour', 'emplois', 'entrees_tunisiens', 'entrees_nationalite', 'etablissements', 'etablissements_region', 'recettes_devises', 'taux_occupation', 'nuitees_nationalite', 'investissements', 'wb_arrivals', 'wb_departures', 'wb_receipts', 'wb_expenditures']


In [8]:

# ============================================================
# 2. ENTRAÎNEMENT + VALIDATION TEMPORELLE + PRÉVISION
# ============================================================
forecasts = {}          # prévisions complètes (historique + futur)
metrics_report = {}     # métriques de validation
models = {}             # modèles entraînés sur tout l'historique

for key, label in INDICATEURS.items():
    print(f"\n{'='*60}")
    print(f"Indicateur : {label} ({key})")

    series = prepare_series(df, key)
    if series.empty or len(series) < 6:
        print(f"  → Série trop courte ou absente. Ignorée.")
        continue

    print(f"  → {len(series)} observations ({series['ds'].min().year} → {series['ds'].max().year})")

    # ----- Validation temporelle -----
    train = series[series["ds"].dt.year <= SPLIT_YEAR].copy()
    test = series[series["ds"].dt.year > SPLIT_YEAR].copy()

    if len(train) < 4 or len(test) == 0:
        print("  → Pas assez de données pour un split train/test. Évaluation in-sample uniquement.")
        model_val = train_prophet(series)
        future_val = model_val.make_future_dataframe(periods=0, freq="YE")
        pred_val = model_val.predict(future_val)
        metrics = compute_metrics(series["y"], pred_val["yhat"])
        print(f"  → Métriques in-sample : {metrics}")
    else:
        model_val = train_prophet(train)
        future_val = model_val.make_future_dataframe(periods=len(test), freq="YE")
        pred_val = model_val.predict(future_val)
        # Alignement sur les dates de test
        pred_test = pred_val.set_index("ds").loc[test["ds"].values]["yhat"]
        metrics = compute_metrics(test["y"].values, pred_test.values)
        print(f"  → Validation temporelle (train ≤ {SPLIT_YEAR}, test > {SPLIT_YEAR})")
        print(f"  → Métriques out-of-sample : {metrics}")

    metrics_report[key] = metrics

    # ----- Modèle final (tout l'historique) pour les prévisions futures -----
    model_final = train_prophet(series, interval_width=0.90)
    future = model_final.make_future_dataframe(periods=5, freq="YE")
    forecast = model_final.predict(future)

    forecasts[key] = forecast
    models[key] = model_final

    # Affichage des prévisions futures
    future_only = forecast[forecast["ds"].dt.year > series["ds"].dt.year.max()]
    print("  → Prévisions 2025-2029 :")
    print(future_only[["ds", "yhat", "yhat_lower", "yhat_upper"]].round(0).to_string(index=False))

    # Sauvegarde individuelle du modèle (CORRECTION DU BUG)
    model_path = os.path.join(MODELS_DIR, f"prophet_{key}.joblib")
    joblib.dump(model_final, model_path)
    print(f"  → Modèle sauvegardé : {model_path}")


Indicateur : Arrivées touristiques (Banque Mondiale) (wb_arrivals)
  → 26 observations (1995 → 2020)


13:32:49 - cmdstanpy - INFO - Chain [1] start processing
13:32:50 - cmdstanpy - INFO - Chain [1] done processing
13:32:50 - cmdstanpy - INFO - Chain [1] start processing


  → Validation temporelle (train ≤ 2019, test > 2019)
  → Métriques out-of-sample : {'MAE': 5851142.74, 'RMSE': np.float64(5851142.74), 'MAPE': np.float64(290.81)}


13:32:50 - cmdstanpy - INFO - Chain [1] done processing
13:32:50 - cmdstanpy - INFO - Chain [1] start processing


  → Prévisions 2025-2029 :
        ds      yhat  yhat_lower  yhat_upper
2021-12-31 7288263.0   5137371.0   9263648.0
2022-12-31 7783517.0   5706683.0   9850185.0
2023-12-31 8225665.0   6125736.0  10256234.0
2024-12-31 7140185.0   5008275.0   9310981.0
2025-12-31 7711253.0   5509345.0   9891462.0
  → Modèle sauvegardé : ../../models\prophet_wb_arrivals.joblib

Indicateur : Recettes touristiques (TND) (recettes_devises)
  → 9 observations (2015 → 2023)


...

  → Prévisions 2025-2029 :
        ds  yhat  yhat_lower  yhat_upper
2024-12-31  18.0        -1.0        37.0
2025-12-31  24.0         4.0        43.0
2026-12-31  30.0         9.0        51.0
2027-12-31  36.0        15.0        54.0
2028-12-31  19.0        -1.0        38.0
  → Modèle sauvegardé : ../../models\prophet_taux_occupation.joblib

Indicateur : Emplois touristiques directs (emplois)
  → 9 observations (2015 → 2023)


13:33:12 - cmdstanpy - INFO - Chain [1] done processing
13:33:12 - cmdstanpy - INFO - Chain [1] start processing


  → Validation temporelle (train ≤ 2019, test > 2019)
  → Métriques out-of-sample : {'MAE': 5.56, 'RMSE': np.float64(5.72), 'MAPE': np.float64(6.04)}


13:33:20 - cmdstanpy - INFO - Chain [1] done processing


  → Prévisions 2025-2029 :
        ds  yhat  yhat_lower  yhat_upper
2024-12-31  90.0        89.0        91.0
2025-12-31  90.0        87.0        93.0
2026-12-31  90.0        85.0        95.0
2027-12-31  90.0        83.0        98.0
2028-12-31  88.0        78.0        99.0
  → Modèle sauvegardé : ../../models\prophet_emplois.joblib


In [9]:
# ============================================================
# 3. RÉSUMÉ DES MÉTRIQUES
# ============================================================
print("\n" + "=" * 70)
print("RÉSUMÉ DES PERFORMANCES (Validation temporelle)")
print("=" * 70)
metrics_df = pd.DataFrame(metrics_report).T
print(metrics_df.to_string())
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "prophet_metrics_report.csv"))
print(f"\n→ Métriques sauvegardées : {OUTPUT_DIR}/prophet_metrics_report.csv")


RÉSUMÉ DES PERFORMANCES (Validation temporelle)
                         MAE        RMSE    MAPE
wb_arrivals       5851142.74  5851142.74  290.81
recettes_devises     2217.31     2309.94   77.22
capacite_lits          96.68      102.78   41.85
taux_occupation        28.30       29.76  221.10
emplois                 5.56        5.72    6.04

→ Métriques sauvegardées : ../../data/processed/prophet_metrics_report.csv


In [10]:

# ============================================================
# 4. EXPORT DES PRÉVISIONS (CSV + JSON)
# ============================================================
summary = []
for key, forecast in forecasts.items():
    # On prend les 5 prochaines années après la dernière observation réelle
    last_real_year = df[df["Dataset"] == key]["Annee"].max()
    future_pred = forecast[forecast["ds"].dt.year > last_real_year].head(5)

    for _, row in future_pred.iterrows():
        summary.append({
            "Annee": int(row["ds"].year),
            "Indicateur": key,
            "Prevision": round(float(row["yhat"]), 1),
            "Lower": round(float(row["yhat_lower"]), 1),
            "Upper": round(float(row["yhat_upper"]), 1),
        })

summary_df = pd.DataFrame(summary)
csv_path = os.path.join(OUTPUT_DIR, "forecast_summary_2025_2029.csv")
summary_df.to_csv(csv_path, index=False)
print(f"→ Prévisions CSV : {csv_path}")

json_path = os.path.join(OUTPUT_DIR, "forecast_summary_2025_2029.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"→ Prévisions JSON : {json_path}")

print("\n" + summary_df.to_string(index=False))
print("\n✅ Module 1 (Prophet) terminé avec succès.")


→ Prévisions CSV : ../../data/processed\forecast_summary_2025_2029.csv
→ Prévisions JSON : ../../data/processed\forecast_summary_2025_2029.json

 Annee       Indicateur  Prevision     Lower      Upper
  2021      wb_arrivals  7288263.4 5137371.0  9263647.7
  2022      wb_arrivals  7783517.4 5706683.2  9850185.3
  2023      wb_arrivals  8225665.0 6125735.9 10256234.4
  2024      wb_arrivals  7140185.1 5008274.5  9310980.9
  2025      wb_arrivals  7711253.5 5509345.1  9891462.1
  2024 recettes_devises     2903.8    1850.2     4047.5
  2025 recettes_devises     4691.1    3656.3     5745.8
  2026 recettes_devises     6685.1    5678.9     7767.7
  2027 recettes_devises     8886.0    7834.8     9912.1
  2028 recettes_devises     3639.8    2570.2     4718.3
  2025    capacite_lits      233.2     230.0      236.4
  2026    capacite_lits      234.3     224.3      244.5
  2027    capacite_lits      235.3     217.3      254.7
  2028    capacite_lits      236.3     208.4      266.7
  2029    capac